# 3. Transformación Silver – Clientes

Objetivo: leer `bronze.clientes`, separar los 12 registros cuyo nombre contiene `DUPLICADO` y preparar:
- `silver.clientes`: 10,000 registros válidos.
- `silver.clientes_rechazados`: 12 registros excluidos con motivo.

La ciudad nula se conserva. Bronze no se modifica. Este notebook define sus propios parámetros y no depende de variables de otros notebooks.

## 1. Inicialización

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import col, upper, trim, lit, current_timestamp

## 2. Parámetros

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaBronze = "bronze"
esquemaSilver = "silver"

ruta_silver = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/silver/"

tabla_clientes_bronze = f"{catalogo}.{esquemaBronze}.clientes"
tabla_clientes_silver = f"{catalogo}.{esquemaSilver}.clientes"
tabla_clientes_rechazados = f"{catalogo}.{esquemaSilver}.clientes_rechazados"

print("Catálogo:", catalogo)
print("Origen:", tabla_clientes_bronze)
print("Destino:", tabla_clientes_silver)
print("Rechazados:", tabla_clientes_rechazados)
print("Ruta Silver:", ruta_silver)

## 3. Lectura desde Bronze

In [ ]:
df_clientes_bronze = spark.table(tabla_clientes_bronze)

print("Registros en Bronze:", df_clientes_bronze.count())
df_clientes_bronze.printSchema()

## 4. Separación de duplicados

Se rechazan los registros cuyo `nombre` contiene `DUPLICADO` (sin distinguir mayúsculas/minúsculas). La separación se realiza en memoria; todavía no se escribe ninguna tabla.

In [ ]:
df_clientes_rechazados = (
    df_clientes_bronze
    .filter(upper(trim(col("nombre"))).contains("DUPLICADO"))
    .withColumn("motivo_rechazo", lit("REGISTRO_DUPLICADO"))
    .withColumn("fecha_proceso", current_timestamp())
)

df_clientes_silver = (
    df_clientes_bronze
    .filter(~upper(trim(col("nombre"))).contains("DUPLICADO"))
)

print("Clientes Bronze:", df_clientes_bronze.count())
print("Clientes Silver:", df_clientes_silver.count())
print("Clientes rechazados:", df_clientes_rechazados.count())

## 5. Validación de conteos y conservación de nulos

In [ ]:
conteo_bronze = df_clientes_bronze.count()
conteo_silver = df_clientes_silver.count()
conteo_rechazados = df_clientes_rechazados.count()

print("Bronze:", conteo_bronze)
print("Silver:", conteo_silver)
print("Rechazados:", conteo_rechazados)
print("Conciliación:", conteo_silver + conteo_rechazados)

assert conteo_bronze == conteo_silver + conteo_rechazados, "Los conteos no concilian."
assert conteo_rechazados == 12, "Se esperaban 12 registros duplicados."
assert conteo_silver == 10000, "Se esperaban 10,000 clientes válidos."

nulos_ciudad_bronze = df_clientes_bronze.filter(col("ciudad").isNull()).count()
nulos_ciudad_silver = df_clientes_silver.filter(col("ciudad").isNull()).count()

print("Ciudades nulas Bronze:", nulos_ciudad_bronze)
print("Ciudades nulas Silver:", nulos_ciudad_silver)

assert nulos_ciudad_bronze == nulos_ciudad_silver, "No se conservaron los nulos de ciudad."
print("Validaciones correctas.")

## 6. Inspección antes de escribir

In [ ]:
display(df_clientes_silver.orderBy("cliente_id").limit(10))

In [ ]:
display(df_clientes_rechazados.orderBy("cliente_id"))

## 7. Preparar esquema Silver

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaSilver}")
print(f"Esquema disponible: {catalogo}.{esquemaSilver}")

## 8. Escritura de tablas Delta

Ejecutar estas celdas solo después de revisar las validaciones. `overwrite` reconstruye las tablas Silver; no modifica Bronze.

In [ ]:
df_clientes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}clientes/") \
    .saveAsTable(tabla_clientes_silver)

print("Tabla creada:", tabla_clientes_silver)

In [ ]:
df_clientes_rechazados.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}clientes_rechazados/") \
    .saveAsTable(tabla_clientes_rechazados)

print("Tabla creada:", tabla_clientes_rechazados)

## 9. Validación final

In [ ]:
print("Clientes Silver:", spark.table(tabla_clientes_silver).count())
print("Clientes rechazados:", spark.table(tabla_clientes_rechazados).count())
spark.table(tabla_clientes_silver).printSchema()